# SARING Price Engine — Colab training notebook

Notebook ini melatih dan mengevaluasi interval harga kondisional untuk laptop COTS.

Yang dijalankan:
1. memuat `price_training_observed.csv.gz` dari upload, Google Drive, atau path;
2. audit schema, kualitas data, dan potensi leakage;
3. out-of-time split;
4. comparable-cohort baseline;
5. LightGBM quantile Q0.50 dan Q0.90 sebagai model utama;
6. CatBoost quantile sebagai challenger dengan trade-off kualitas versus waktu/ukuran;
7. pinball loss, Q0.90 coverage, calibration slices, dan product-disjoint stress test;
8. abstention gate, contoh scoring, dan export artefak.

Batas klaim: model mempelajari perilaku harga terobservasi. Output bukan fraud probability, bukan kerugian negara, dan bukan keputusan pengadaan otomatis. Nilai di atas Q90 adalah deviasi harga yang membutuhkan penjelasan dan verifikasi manusia.


In [ ]:
# Jalankan sekali pada runtime Colab baru.
%pip install -q lightgbm==4.7.0 catboost==1.2.10


In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import math
import os
import platform
import shutil
import time
import warnings
from pathlib import Path

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from catboost import CatBoostRegressor
from sklearn.metrics import mean_pinball_loss

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid")

SEED = 20260808
np.random.seed(SEED)

# Pilih: "UPLOAD", "DRIVE", atau "PATH".
DATA_MODE = "UPLOAD"
DATA_PATH = "/content/price_training_observed.csv.gz"
DRIVE_PATH = "/content/drive/MyDrive/SARING/price_training_observed.csv.gz"

MIN_COMPARATORS = 8
HOLDOUT_FRACTION = 0.20
USE_LOG_TARGET = True
RUN_CATBOOST = True
RUN_PRODUCT_DISJOINT_STRESS = True
STRICT_OPERATIONAL_GATE = True
COVERAGE_TOLERANCE = 0.05

ARTIFACT_DIR = Path("/content/saring_price_engine_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "lightgbm": lgb.__version__,
    "data_mode": DATA_MODE,
})


## 1. Load observed price rows

Dataset yang benar adalah export observed-only. Jangan gunakan controlled scenario targets, winner/award labels, evaluation outcome, contract, payment, fraud/scenario labels, atau evidence yang muncul setelah analysis cutoff sebagai feature.


In [ ]:
def resolve_data_path() -> Path:
    mode = DATA_MODE.upper()
    if mode == "UPLOAD":
        try:
            from google.colab import files
        except ImportError as exc:
            raise RuntimeError("UPLOAD mode hanya tersedia di Google Colab. Gunakan DATA_MODE='PATH'.") from exc
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("Tidak ada file yang di-upload.")
        candidates = [Path("/content") / name for name in uploaded]
        preferred = [p for p in candidates if p.name.endswith((".csv", ".csv.gz", ".parquet"))]
        if not preferred:
            raise ValueError("Upload CSV, CSV.GZ, atau Parquet.")
        return preferred[0]
    if mode == "DRIVE":
        from google.colab import drive
        drive.mount("/content/drive")
        return Path(DRIVE_PATH)
    if mode == "PATH":
        return Path(DATA_PATH)
    raise ValueError("DATA_MODE harus UPLOAD, DRIVE, atau PATH.")


def read_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    suffixes = "".join(path.suffixes).lower()
    if suffixes.endswith(".parquet"):
        return pd.read_parquet(path)
    if suffixes.endswith(".csv") or suffixes.endswith(".csv.gz"):
        return pd.read_csv(path, low_memory=False)
    raise ValueError(f"Format belum didukung: {path.name}")


SOURCE_PATH = resolve_data_path()
RAW_SHA256 = hashlib.sha256(SOURCE_PATH.read_bytes()).hexdigest()
raw = read_table(SOURCE_PATH)

print(f"Source: {SOURCE_PATH}")
print(f"SHA-256: {RAW_SHA256}")
print(f"Rows x columns: {raw.shape}")
display(raw.head())


## 2. Map source columns to the canonical training view

Loader mengenali alias umum. Jika field wajib gagal ditemukan, tambahkan nama kolom dataset Anda ke COLUMN_ALIASES. Hanya unit_price dan observed_at yang wajib untuk training; field comparability yang hilang tetap dicatat dan dapat menyebabkan abstention operasional.


In [ ]:
COLUMN_ALIASES = {
    "record_id": ["record_id", "observation_id", "price_observation_id", "source_record_id", "item_id"],
    "unit_price": ["unit_price", "observed_unit_price", "price", "price_amount", "unit_price_observed"],
    "observed_at": ["observed_at", "observation_time", "event_at", "procurement_date", "captured_at", "date"],
    "analysis_cutoff_at": ["analysis_cutoff_at", "cutoff_at"],
    "manufacturer_part_number": ["manufacturer_part_number", "part_number", "mpn", "sku"],
    "brand": ["brand", "manufacturer", "make"],
    "model_name": ["model_name", "model", "product_model"],
    "cpu": ["cpu", "cpu_model", "processor", "processor_model"],
    "cpu_family": ["cpu_family", "processor_family"],
    "cpu_generation": ["cpu_generation", "processor_generation"],
    "ram_gb": ["ram_gb", "memory_gb", "ram_capacity_gb"],
    "storage": ["storage", "storage_spec", "storage_type"],
    "storage_gb": ["storage_gb", "storage_capacity_gb", "disk_gb"],
    "display": ["display", "display_spec", "screen"],
    "display_size_inch": ["display_size_inch", "screen_size_inch", "display_inches"],
    "display_resolution": ["display_resolution", "screen_resolution", "resolution"],
    "operating_system": ["operating_system", "os"],
    "condition": ["condition", "item_condition", "product_condition"],
    "quantity": ["quantity", "qty"],
    "unit": ["unit", "unit_of_measure", "uom"],
    "currency": ["currency", "currency_code"],
    "tax_included": ["tax_included", "includes_tax", "tax_basis"],
    "shipping_included": ["shipping_included", "includes_shipping", "shipping_basis"],
    "installation_included": ["installation_included", "includes_installation", "installation_basis"],
    "warranty_months": ["warranty_months", "warranty_duration_months", "warranty"],
    "region_code": ["region_code", "region", "location_code"],
    "service_bundle": ["service_bundle", "service_components", "bundle_type"],
    "cpu_cores": ["cpu_cores", "processor_cores"],
    "cpu_threads": ["cpu_threads", "processor_threads"],
    "ram_type": ["ram_type", "memory_type"],
    "storage_type": ["storage_type", "disk_type"],
    "display_panel": ["display_panel", "panel_type"],
    "delivery_days": ["delivery_days", "lead_time_days"],
    "region_group": ["region_group", "region_tier"],
    "seller_tier": ["seller_tier", "provider_tier"],
    "source_system": ["source_system", "source", "provider"],
    "source_product_id": ["source_product_id", "product_id"],
    "category_code": ["category_code", "category", "commodity_code"],
}

REQUIRED_FIELDS = {"unit_price", "observed_at"}


def resolve_column_mapping(frame: pd.DataFrame) -> dict[str, str]:
    lookup = {str(c).strip().lower(): c for c in frame.columns}
    mapping = {}
    for canonical, aliases in COLUMN_ALIASES.items():
        matches = [lookup[a.lower()] for a in aliases if a.lower() in lookup]
        if matches:
            mapping[canonical] = matches[0]
    missing = REQUIRED_FIELDS - mapping.keys()
    if missing:
        raise ValueError(
            f"Field wajib tidak ditemukan: {sorted(missing)}. "
            f"Kolom tersedia: {list(frame.columns)}. Update COLUMN_ALIASES."
        )
    return mapping


COLUMN_MAPPING = resolve_column_mapping(raw)
df = pd.DataFrame(index=raw.index)
for canonical in COLUMN_ALIASES:
    source = COLUMN_MAPPING.get(canonical)
    df[canonical] = raw[source] if source is not None else np.nan

print("Resolved mapping:")
display(pd.Series(COLUMN_MAPPING, name="source_column").to_frame())


## 3. Quality checks, normalization, and leakage guard

The feature allowlist below is deliberate. Suspicious columns may remain in the raw file for audit, but they can never enter model features.


In [ ]:
LEAKAGE_PATTERNS = (
    "winner", "award", "evaluation", "disqual", "target", "scenario", "fraud",
    "corrupt", "collusion", "contract", "payment", "state_loss", "post_cutoff",
)
suspicious_raw_columns = [
    c for c in raw.columns
    if any(token in str(c).lower() for token in LEAKAGE_PATTERNS)
]
print("Suspicious raw columns excluded from features:", suspicious_raw_columns)

initial_rows = len(df)
df["observed_at"] = pd.to_datetime(df["observed_at"], errors="coerce", utc=True)
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

if df["analysis_cutoff_at"].notna().any():
    df["analysis_cutoff_at"] = pd.to_datetime(df["analysis_cutoff_at"], errors="coerce", utc=True)
    post_cutoff = df["analysis_cutoff_at"].notna() & (df["observed_at"] > df["analysis_cutoff_at"])
else:
    post_cutoff = pd.Series(False, index=df.index)

invalid_price = df["unit_price"].isna() | ~np.isfinite(df["unit_price"]) | (df["unit_price"] <= 0)
invalid_date = df["observed_at"].isna()
df = df.loc[~invalid_price & ~invalid_date & ~post_cutoff].copy()

for col in ["ram_gb", "storage_gb", "display_size_inch", "quantity", "warranty_months"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in [
    "record_id", "manufacturer_part_number", "brand", "model_name", "cpu",
    "cpu_family", "cpu_generation", "storage", "display", "display_resolution",
    "operating_system", "condition", "unit", "currency", "tax_included",
    "shipping_included", "installation_included", "region_code", "service_bundle",
    "source_system", "source_product_id", "category_code",
]:
    df[col] = df[col].astype("string").str.strip()
    df.loc[df[col].str.lower().isin(["", "nan", "none", "null", "unknown", "n/a"]), col] = pd.NA

df["observed_year"] = df["observed_at"].dt.year.astype(float)
df["observed_month"] = df["observed_at"].dt.month.astype(float)
df["observed_quarter"] = df["observed_at"].dt.quarter.astype(float)
df["time_index_days"] = (
    (df["observed_at"] - df["observed_at"].min()).dt.total_seconds() / 86400
)

quality_report = pd.DataFrame({
    "value": {
        "raw_rows": initial_rows,
        "usable_rows": len(df),
        "invalid_price_rows": int(invalid_price.sum()),
        "invalid_date_rows": int(invalid_date.sum()),
        "post_cutoff_rows_excluded": int(post_cutoff.sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "date_min": str(df["observed_at"].min()),
        "date_max": str(df["observed_at"].max()),
        "price_min": float(df["unit_price"].min()),
        "price_median": float(df["unit_price"].median()),
        "price_max": float(df["unit_price"].max()),
    }
})
display(quality_report)

if len(df) < 500:
    warnings.warn("Dataset sangat kecil untuk gradient boosting; interpretasikan hasil sebagai eksperimen.")


In [ ]:
NUMERIC_CANDIDATES = [
    "ram_gb", "storage_gb", "display_size_inch", "quantity", "warranty_months",
    "cpu_cores", "cpu_threads", "delivery_days",
    "observed_year", "observed_month", "observed_quarter", "time_index_days",
]
CATEGORICAL_CANDIDATES = [
    "manufacturer_part_number", "brand", "model_name", "cpu", "cpu_family",
    "cpu_generation", "storage", "display", "display_resolution",
    "operating_system", "condition", "unit", "currency", "tax_included",
    "shipping_included", "installation_included", "region_code",
    "service_bundle", "ram_type", "storage_type", "display_panel",
    "region_group", "seller_tier", "source_system", "category_code",
]

NUMERIC_FEATURES = [c for c in NUMERIC_CANDIDATES if df[c].notna().any()]
CATEGORICAL_FEATURES = [c for c in CATEGORICAL_CANDIDATES if df[c].notna().any()]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

forbidden_features = [
    c for c in FEATURES
    if any(token in c.lower() for token in LEAKAGE_PATTERNS)
]
assert not forbidden_features, f"Leakage feature terdeteksi: {forbidden_features}"
assert len(FEATURES) >= 2, "Feature usable terlalu sedikit."

missingness = (
    df[FEATURES].isna().mean().sort_values(ascending=False)
    .rename("missing_fraction").to_frame()
)
print(f"{len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical features")
display(missingness)


## 4. Out-of-time holdout

Baris paling baru menjadi test set. Split acak tidak dipakai untuk keputusan model karena dapat memberi estimasi terlalu optimistis ketika level harga dan katalog berubah sepanjang waktu.


In [ ]:
ordered_index = df.sort_values(["observed_at", "record_id"], na_position="last").index
test_cut = int(len(ordered_index) * (1 - HOLDOUT_FRACTION))
test_cut = min(max(test_cut, 2), len(ordered_index) - 1)
pretest_idx = ordered_index[:test_cut]
test_idx = ordered_index[test_cut:]

# Validation is the newest 20% of the pre-test window. It is used for early
# stopping and model choice; the final out-of-time test stays untouched.
valid_cut = int(len(pretest_idx) * 0.80)
valid_cut = min(max(valid_cut, 1), len(pretest_idx) - 1)
train_idx = pretest_idx[:valid_cut]
valid_idx = pretest_idx[valid_cut:]

train_df = df.loc[train_idx].copy()
valid_df = df.loc[valid_idx].copy()
test_df = df.loc[test_idx].copy()

y_train_raw = train_df["unit_price"].to_numpy(float)
y_valid = valid_df["unit_price"].to_numpy(float)
y_test = test_df["unit_price"].to_numpy(float)
y_train = np.log1p(y_train_raw) if USE_LOG_TARGET else y_train_raw

split_report = pd.DataFrame({
    "rows": [len(train_df), len(valid_df), len(test_df)],
    "date_min": [
        train_df["observed_at"].min(), valid_df["observed_at"].min(), test_df["observed_at"].min()
    ],
    "date_max": [
        train_df["observed_at"].max(), valid_df["observed_at"].max(), test_df["observed_at"].max()
    ],
    "median_price": [
        train_df["unit_price"].median(), valid_df["unit_price"].median(), test_df["unit_price"].median()
    ],
}, index=["train", "validation", "final_out_of_time_test"])
display(split_report)


## 5. Defensible comparable cohort and transparent baseline

Cohort dipilih secara berjenjang: exact part number, strict configuration, lalu relaxed configuration. Level pertama yang mencapai minimum comparator digunakan. Jika tidak ada level yang cukup, observasi berstatus abstained untuk penggunaan operasional.


In [ ]:
COHORT_LEVELS = [
    ("EXACT_SKU", ["manufacturer_part_number"]),
    ("STRICT_CONFIGURATION", ["brand", "cpu", "ram_gb", "storage", "condition"]),
    ("RELAXED_CONFIGURATION", ["cpu", "ram_gb", "storage", "condition"]),
]


def normalized_key_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        return value.strip().lower()
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return round(float(value), 4)
    return str(value).strip().lower()


def make_key(row: pd.Series, columns: list[str]):
    values = tuple(normalized_key_value(row[c]) for c in columns)
    if any(v is None for v in values):
        return None
    return values


def build_cohort_maps(reference: pd.DataFrame):
    maps = {}
    for name, columns in COHORT_LEVELS:
        groups = {}
        for idx, row in reference.iterrows():
            key = make_key(row, columns)
            if key is not None:
                groups.setdefault(key, []).append(idx)
        maps[name] = (columns, groups)
    return maps


COHORT_MAPS = build_cohort_maps(train_df)


def find_comparable_indices(row: pd.Series, min_count: int = MIN_COMPARATORS):
    best_name, best_indices = "NONE", []
    for name, (columns, groups) in COHORT_MAPS.items():
        key = make_key(row, columns)
        indices = [] if key is None else groups.get(key, [])
        if len(indices) > len(best_indices):
            best_name, best_indices = name, indices
        if len(indices) >= min_count:
            return name, indices, True
    return best_name, best_indices, False


def evaluate_comparable_cohorts(target_frame: pd.DataFrame):
    cohort_rows = []
    q50 = np.full(len(target_frame), np.nan)
    q90 = np.full(len(target_frame), np.nan)
    for position, (_, row) in enumerate(target_frame.iterrows()):
        level, indices, sufficient = find_comparable_indices(row)
        prices = train_df.loc[indices, "unit_price"].to_numpy(float) if indices else np.array([])
        if sufficient:
            q50[position] = np.quantile(prices, 0.50)
            q90[position] = np.quantile(prices, 0.90)
        cohort_rows.append({
            "cohort_level": level,
            "comparator_count": len(indices),
            "cohort_sufficient": sufficient,
        })
    return pd.DataFrame(cohort_rows, index=target_frame.index), q50, q90


valid_cohort_eval, valid_baseline_q50, valid_baseline_q90 = evaluate_comparable_cohorts(valid_df)
cohort_eval, baseline_q50, baseline_q90 = evaluate_comparable_cohorts(test_df)

display(cohort_eval["cohort_level"].value_counts(dropna=False).to_frame("final_test_rows"))
print("Final-test comparator-sufficient rate:", f"{cohort_eval['cohort_sufficient'].mean():.1%}")


## 6. Operational abstention gate

Training/evaluation statistik tetap dapat berjalan ketika sebagian basis context hilang. Scoring operasional lebih ketat: product/spec resolved, comparator count cukup, unit/currency valid, serta basis tax/shipping/warranty diketahui. Ubah STRICT_OPERATIONAL_GATE hanya untuk eksperimen, bukan untuk menyembunyikan data yang tidak lengkap.


In [ ]:
def known(value) -> bool:
    return not pd.isna(value) and str(value).strip().lower() not in {"", "unknown", "nan", "none", "null", "n/a"}


def abstention_reasons(row: pd.Series, comparator_count: int) -> list[str]:
    reasons = []
    resolved_spec_count = sum(known(row.get(c)) for c in ["brand", "cpu", "ram_gb", "storage"])
    if not known(row.get("manufacturer_part_number")) and resolved_spec_count < 3:
        reasons.append("SKU/specification cannot be resolved")
    if comparator_count < MIN_COMPARATORS:
        reasons.append("insufficient comparable cohort")
    if not known(row.get("unit")):
        reasons.append("target unit is unresolved")
    if not known(row.get("currency")):
        reasons.append("target currency is unresolved")
    if STRICT_OPERATIONAL_GATE:
        if not known(row.get("tax_included")):
            reasons.append("tax basis is unknown")
        if not known(row.get("shipping_included")):
            reasons.append("shipping basis is unknown")
        if not known(row.get("warranty_months")):
            reasons.append("warranty basis is unknown")
    return reasons


def apply_operational_gate(target_frame: pd.DataFrame, cohort_frame: pd.DataFrame):
    reasons = []
    for position, (_, row) in enumerate(target_frame.iterrows()):
        reasons.append(abstention_reasons(row, int(cohort_frame.iloc[position]["comparator_count"])))
    result = cohort_frame.copy()
    result["abstention_reasons"] = reasons
    result["operationally_eligible"] = result["abstention_reasons"].map(len).eq(0)
    return result


valid_cohort_eval = apply_operational_gate(valid_df, valid_cohort_eval)
cohort_eval = apply_operational_gate(test_df, cohort_eval)
gate_reasons = cohort_eval["abstention_reasons"].tolist()

print("Final-test operationally eligible:", f"{cohort_eval['operationally_eligible'].mean():.1%}")
display(
    pd.Series([r for reasons in gate_reasons for r in reasons], dtype="string")
    .value_counts().rename_axis("reason").to_frame("rows")
)


## 7. Train primary model: LightGBM quantile Q0.50 and Q0.90

Categorical levels and numeric medians are fit only on the training window. Unknown categories at inference are mapped explicitly.


In [ ]:
def fit_preprocessor(reference: pd.DataFrame):
    numeric_medians = {
        col: float(pd.to_numeric(reference[col], errors="coerce").median())
        if pd.to_numeric(reference[col], errors="coerce").notna().any() else 0.0
        for col in NUMERIC_FEATURES
    }
    category_levels = {}
    for col in CATEGORICAL_FEATURES:
        values = reference[col].astype("string").fillna("__MISSING__")
        levels = sorted(set(values.tolist()) | {"__MISSING__", "__UNKNOWN__"})
        category_levels[col] = levels
    return numeric_medians, category_levels


def transform_lgbm(frame: pd.DataFrame, numeric_medians, category_levels):
    out = pd.DataFrame(index=frame.index)
    for col in NUMERIC_FEATURES:
        out[col] = pd.to_numeric(frame[col], errors="coerce").fillna(numeric_medians[col]).astype(float)
    for col in CATEGORICAL_FEATURES:
        values = frame[col].astype("string").fillna("__MISSING__")
        levels = category_levels[col]
        values = values.where(values.isin(levels), "__UNKNOWN__")
        out[col] = pd.Categorical(values, categories=levels)
    return out[FEATURES]


NUMERIC_MEDIANS, CATEGORY_LEVELS = fit_preprocessor(train_df)
X_train_lgb = transform_lgbm(train_df, NUMERIC_MEDIANS, CATEGORY_LEVELS)
X_valid_lgb = transform_lgbm(valid_df, NUMERIC_MEDIANS, CATEGORY_LEVELS)
X_test_lgb = transform_lgbm(test_df, NUMERIC_MEDIANS, CATEGORY_LEVELS)


def train_lgbm_quantiles(X_fit, y_fit, X_valid, y_valid, seed=SEED):
    models = {}
    start = time.perf_counter()
    for alpha in (0.50, 0.90):
        model = lgb.LGBMRegressor(
            objective="quantile",
            alpha=alpha,
            n_estimators=3000,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=30,
            subsample=0.90,
            subsample_freq=1,
            colsample_bytree=0.90,
            reg_alpha=0.10,
            reg_lambda=1.00,
            random_state=seed,
            n_jobs=-1,
            verbosity=-1,
        )
        model.fit(
            X_fit,
            y_fit,
            eval_set=[(X_valid, y_valid)],
            eval_metric="quantile",
            categorical_feature=CATEGORICAL_FEATURES,
            callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(0)],
        )
        models[alpha] = model
        print(f"Q{int(alpha * 100)} best iteration: {model.best_iteration_}")
    return models, time.perf_counter() - start


lgb_models, lgb_train_seconds = train_lgbm_quantiles(
    X_train_lgb,
    y_train,
    X_valid_lgb,
    np.log1p(y_valid) if USE_LOG_TARGET else y_valid,
)

def invert_target(values):
    values = np.asarray(values, dtype=float)
    return np.expm1(values) if USE_LOG_TARGET else values

lgb_valid_q50_raw = invert_target(lgb_models[0.50].predict(X_valid_lgb))
lgb_valid_q90_raw = invert_target(lgb_models[0.90].predict(X_valid_lgb))
lgb_valid_q50 = np.maximum(lgb_valid_q50_raw, 0)
lgb_valid_q90 = np.maximum.reduce([lgb_valid_q90_raw, lgb_valid_q50, np.zeros_like(lgb_valid_q50)])

lgb_q50_raw = invert_target(lgb_models[0.50].predict(X_test_lgb))
lgb_q90_raw = invert_target(lgb_models[0.90].predict(X_test_lgb))
lgb_crossing_rate = float(np.mean(lgb_q90_raw < lgb_q50_raw))
lgb_q50 = np.maximum(lgb_q50_raw, 0)
lgb_q90 = np.maximum.reduce([lgb_q90_raw, lgb_q50, np.zeros_like(lgb_q50)])

print(f"LightGBM training time: {lgb_train_seconds:.1f}s")
print(f"Raw quantile crossing rate: {lgb_crossing_rate:.2%}")


## 8. Train challenger: CatBoost quantile

CatBoost sering kuat pada data tabular kategorikal dengan preprocessing minimal. Trade-off-nya adalah package/model yang lebih besar dan training yang biasanya lebih lambat. Challenger hanya dipilih bila peningkatan out-of-time cukup berarti.


In [ ]:
def fit_catboost_frame(reference: pd.DataFrame):
    numeric_medians = {
        col: float(pd.to_numeric(reference[col], errors="coerce").median())
        if pd.to_numeric(reference[col], errors="coerce").notna().any() else 0.0
        for col in NUMERIC_FEATURES
    }
    return numeric_medians


def transform_catboost(frame: pd.DataFrame, numeric_medians):
    out = pd.DataFrame(index=frame.index)
    for col in NUMERIC_FEATURES:
        out[col] = pd.to_numeric(frame[col], errors="coerce").fillna(numeric_medians[col]).astype(float)
    for col in CATEGORICAL_FEATURES:
        out[col] = frame[col].astype("string").fillna("__MISSING__").astype(str)
    return out[FEATURES]


CAT_NUMERIC_MEDIANS = fit_catboost_frame(train_df)
X_train_cat = transform_catboost(train_df, CAT_NUMERIC_MEDIANS)
X_valid_cat = transform_catboost(valid_df, CAT_NUMERIC_MEDIANS)
X_test_cat = transform_catboost(test_df, CAT_NUMERIC_MEDIANS)
cat_feature_indices = [FEATURES.index(c) for c in CATEGORICAL_FEATURES]

cat_models = {}
cat_train_seconds = np.nan
cat_q50 = cat_q90 = None
cat_crossing_rate = np.nan

if RUN_CATBOOST:
    start = time.perf_counter()
    for alpha in (0.50, 0.90):
        model = CatBoostRegressor(
            loss_function=f"Quantile:alpha={alpha}",
            eval_metric=f"Quantile:alpha={alpha}",
            iterations=2500,
            depth=8,
            learning_rate=0.04,
            l2_leaf_reg=5.0,
            random_seed=SEED,
            allow_writing_files=False,
            verbose=False,
            thread_count=-1,
        )
        model.fit(
            X_train_cat,
            y_train,
            cat_features=cat_feature_indices,
            eval_set=(X_valid_cat, np.log1p(y_valid) if USE_LOG_TARGET else y_valid),
            early_stopping_rounds=150,
            verbose=250,
        )
        cat_models[alpha] = model
        print(f"CatBoost Q{int(alpha * 100)} best iteration: {model.get_best_iteration()}")
    cat_train_seconds = time.perf_counter() - start
    cat_valid_q50_raw = invert_target(cat_models[0.50].predict(X_valid_cat))
    cat_valid_q90_raw = invert_target(cat_models[0.90].predict(X_valid_cat))
    cat_valid_q50 = np.maximum(cat_valid_q50_raw, 0)
    cat_valid_q90 = np.maximum.reduce([cat_valid_q90_raw, cat_valid_q50, np.zeros_like(cat_valid_q50)])

    cat_q50_raw = invert_target(cat_models[0.50].predict(X_test_cat))
    cat_q90_raw = invert_target(cat_models[0.90].predict(X_test_cat))
    cat_crossing_rate = float(np.mean(cat_q90_raw < cat_q50_raw))
    cat_q50 = np.maximum(cat_q50_raw, 0)
    cat_q90 = np.maximum.reduce([cat_q90_raw, cat_q50, np.zeros_like(cat_q50)])
    print(f"CatBoost training time: {cat_train_seconds:.1f}s")
    print(f"Raw quantile crossing rate: {cat_crossing_rate:.2%}")
else:
    print("CatBoost skipped (RUN_CATBOOST=False).")


## 9. Apples-to-apples out-of-time evaluation and model selection

Semua model dibandingkan pada test rows dengan comparable cohort yang cukup. Operational gate ditampilkan terpisah agar missing tax/shipping/warranty tidak disamarkan sebagai keberhasilan model.


In [ ]:
def metric_row(name, y_true, q50, q90, mask, train_seconds=np.nan, crossing_rate=np.nan):
    mask = np.asarray(mask, dtype=bool).copy()
    mask &= np.isfinite(q50) & np.isfinite(q90) & np.isfinite(y_true)
    if mask.sum() == 0:
        return {
            "model": name, "rows": 0, "pinball_q50": np.nan, "pinball_q90": np.nan,
            "q90_coverage": np.nan, "coverage_error": np.nan,
            "normalized_pinball": np.nan, "train_seconds": train_seconds,
            "crossing_rate_raw": crossing_rate,
        }
    yt, p50, p90 = y_true[mask], np.asarray(q50)[mask], np.asarray(q90)[mask]
    pb50 = mean_pinball_loss(yt, p50, alpha=0.50)
    pb90 = mean_pinball_loss(yt, p90, alpha=0.90)
    scale = max(float(np.median(yt)), 1.0)
    coverage = float(np.mean(yt <= p90))
    return {
        "model": name,
        "rows": int(mask.sum()),
        "pinball_q50": float(pb50),
        "pinball_q90": float(pb90),
        "q90_coverage": coverage,
        "coverage_error": abs(coverage - 0.90),
        "normalized_pinball": float((pb50 + pb90) / scale),
        "train_seconds": train_seconds,
        "crossing_rate_raw": crossing_rate,
    }


valid_comparable_mask = valid_cohort_eval["cohort_sufficient"].to_numpy(bool)
comparable_mask = cohort_eval["cohort_sufficient"].to_numpy(bool)
operational_mask = cohort_eval["operationally_eligible"].to_numpy(bool)

validation_rows = [
    metric_row("cohort_baseline", y_valid, valid_baseline_q50, valid_baseline_q90, valid_comparable_mask, 0.0, 0.0),
    metric_row("lightgbm_quantile", y_valid, lgb_valid_q50, lgb_valid_q90, valid_comparable_mask, lgb_train_seconds, lgb_crossing_rate),
]
if RUN_CATBOOST:
    validation_rows.append(metric_row(
        "catboost_quantile", y_valid, cat_valid_q50, cat_valid_q90,
        valid_comparable_mask, cat_train_seconds, cat_crossing_rate,
    ))
validation_benchmark = pd.DataFrame(validation_rows).set_index("model")
validation_benchmark["quality_score"] = (
    validation_benchmark["normalized_pinball"] + validation_benchmark["coverage_error"]
)

selected_model_name = "lightgbm_quantile"
selection_reason = "LightGBM is the primary learned candidate."
if RUN_CATBOOST and validation_benchmark.loc["catboost_quantile", "rows"] > 0:
    lgb_quality = validation_benchmark.loc["lightgbm_quantile", "quality_score"]
    cat_quality = validation_benchmark.loc["catboost_quantile", "quality_score"]
    relative_gain = (lgb_quality - cat_quality) / max(lgb_quality, 1e-12)
    if relative_gain >= 0.02:
        selected_model_name = "catboost_quantile"
        selection_reason = (
            f"CatBoost became the learned candidate on validation: quality score improved {relative_gain:.1%}."
        )
    else:
        selection_reason = (
            f"LightGBM remained the learned candidate: CatBoost validation gain ({relative_gain:.1%}) "
            "was below the 2% materiality threshold."
        )

# A learned model is not justified merely because it trained successfully.
# Fall back to the transparent comparable-cohort baseline unless the learned
# candidate improves validation quality by at least 2%.
baseline_quality = validation_benchmark.loc["cohort_baseline", "quality_score"]
candidate_quality = validation_benchmark.loc[selected_model_name, "quality_score"]
if pd.notna(baseline_quality) and pd.notna(candidate_quality):
    advanced_gain = (baseline_quality - candidate_quality) / max(baseline_quality, 1e-12)
    if advanced_gain < 0.02:
        learned_candidate = selected_model_name
        selected_model_name = "cohort_baseline"
        selection_reason = (
            f"Transparent cohort baseline retained: {learned_candidate} validation gain "
            f"({advanced_gain:.1%}) was below the 2% justification threshold."
        )

test_rows = [
    metric_row("cohort_baseline", y_test, baseline_q50, baseline_q90, comparable_mask, 0.0, 0.0),
    metric_row("lightgbm_quantile", y_test, lgb_q50, lgb_q90, comparable_mask, lgb_train_seconds, lgb_crossing_rate),
]
if RUN_CATBOOST:
    test_rows.append(metric_row(
        "catboost_quantile", y_test, cat_q50, cat_q90,
        comparable_mask, cat_train_seconds, cat_crossing_rate,
    ))
benchmark = pd.DataFrame(test_rows).set_index("model")
benchmark["quality_score"] = benchmark["normalized_pinball"] + benchmark["coverage_error"]

operational_rows = [
    metric_row("lightgbm_quantile", y_test, lgb_q50, lgb_q90, operational_mask, lgb_train_seconds, lgb_crossing_rate)
]
if RUN_CATBOOST:
    operational_rows.append(metric_row(
        "catboost_quantile", y_test, cat_q50, cat_q90,
        operational_mask, cat_train_seconds, cat_crossing_rate,
    ))
operational_benchmark = pd.DataFrame(operational_rows).set_index("model")

print("Validation metrics used for model selection:")
display(validation_benchmark.sort_values("quality_score"))
print("Untouched final out-of-time test metrics:")
display(benchmark.sort_values("quality_score"))
print("Final-test operationally gated metrics (may be empty):")
display(operational_benchmark)
print(selected_model_name)
print(selection_reason)


In [ ]:
evaluation = test_df[[
    "record_id", "observed_at", "unit_price", "region_code", "category_code",
    "manufacturer_part_number", "brand", "model_name",
]].copy()
evaluation = evaluation.join(cohort_eval)
evaluation["baseline_q50"] = baseline_q50
evaluation["baseline_q90"] = baseline_q90
evaluation["lgb_q50"] = lgb_q50
evaluation["lgb_q90"] = lgb_q90
if RUN_CATBOOST:
    evaluation["cat_q50"] = cat_q50
    evaluation["cat_q90"] = cat_q90

if selected_model_name == "catboost_quantile":
    selected_q50, selected_q90 = cat_q50, cat_q90
elif selected_model_name == "cohort_baseline":
    selected_q50, selected_q90 = baseline_q50, baseline_q90
else:
    selected_q50, selected_q90 = lgb_q50, lgb_q90
evaluation["selected_q50"] = selected_q50
evaluation["selected_q90"] = selected_q90
evaluation["above_q90"] = evaluation["unit_price"] > evaluation["selected_q90"]
evaluation["relative_deviation"] = np.maximum(
    0, (evaluation["unit_price"] - evaluation["selected_q90"]) / evaluation["selected_q90"]
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_mask = comparable_mask & np.isfinite(selected_q90)
axes[0].scatter(
    evaluation.loc[plot_mask, "unit_price"],
    evaluation.loc[plot_mask, "selected_q90"],
    alpha=0.45,
)
limit = float(np.nanpercentile(
    np.r_[evaluation.loc[plot_mask, "unit_price"], evaluation.loc[plot_mask, "selected_q90"]], 99
)) if plot_mask.any() else 1
axes[0].plot([0, limit], [0, limit], "--", color="black")
axes[0].set(xlabel="Observed unit price", ylabel="Predicted Q90", title="Out-of-time Q90")

if plot_mask.any():
    banded = evaluation.loc[plot_mask].assign(
        price_band=lambda x: pd.qcut(x["unit_price"], q=4, duplicates="drop")
    )
    coverage_by_price_band = banded.groupby("price_band", observed=True).agg(
        rows=("unit_price", "size")
    )
    coverage_by_price_band["q90_coverage"] = banded.groupby("price_band", observed=True).apply(
        lambda g: (g["unit_price"] <= g["selected_q90"]).mean()
    )
    coverage_by_price_band["q90_coverage"].plot.bar(ax=axes[1], color="#155EEF")
    axes[1].axhline(0.90, linestyle="--", color="black")
else:
    coverage_by_price_band = pd.DataFrame(columns=["rows", "q90_coverage"])
    axes[1].text(0.5, 0.5, "No comparator-sufficient rows", ha="center", va="center")
axes[1].set(title="Q90 coverage by observed price band", ylabel="Coverage", xlabel="")
plt.tight_layout()
plt.show()
display(coverage_by_price_band)


## 10. Product-disjoint stress test

Stress test ini menahan seluruh product group dari training. Ia bukan pengganti out-of-time test, tetapi mengukur risiko ketika model menghadapi SKU/configuration yang belum pernah dilihat.


In [ ]:
def product_group_key(row: pd.Series) -> str:
    if known(row.get("manufacturer_part_number")):
        return "sku|" + str(row["manufacturer_part_number"]).strip().lower()
    parts = [normalized_key_value(row.get(c)) for c in ["brand", "cpu", "ram_gb", "storage"]]
    return "cfg|" + "|".join("<missing>" if p is None else str(p) for p in parts)


df["product_group"] = df.apply(product_group_key, axis=1)
unique_groups = sorted(df["product_group"].dropna().unique())
stress_test_groups = {
    group for group in unique_groups
    if int(hashlib.sha256(group.encode()).hexdigest()[:8], 16) % 5 == 0
}
stress_train_df = df.loc[~df["product_group"].isin(stress_test_groups)].copy()
stress_test_df = df.loc[df["product_group"].isin(stress_test_groups)].copy()

stress_result = {
    "train_rows": len(stress_train_df),
    "test_rows": len(stress_test_df),
    "train_groups": stress_train_df["product_group"].nunique(),
    "test_groups": stress_test_df["product_group"].nunique(),
    "tested_selected_model": selected_model_name,
}

if RUN_PRODUCT_DISJOINT_STRESS and len(stress_train_df) >= 200 and len(stress_test_df) >= 50:
    y_stress_train_raw = stress_train_df["unit_price"].to_numpy(float)
    y_stress_test = stress_test_df["unit_price"].to_numpy(float)
    y_stress_train = np.log1p(y_stress_train_raw) if USE_LOG_TARGET else y_stress_train_raw
    y_stress_valid = np.log1p(y_stress_test) if USE_LOG_TARGET else y_stress_test

    if selected_model_name == "catboost_quantile":
        stress_medians = fit_catboost_frame(stress_train_df)
        X_stress_train = transform_catboost(stress_train_df, stress_medians)
        X_stress_test = transform_catboost(stress_test_df, stress_medians)
        stress_models = {}
        start = time.perf_counter()
        for alpha in (0.50, 0.90):
            model = CatBoostRegressor(
                loss_function=f"Quantile:alpha={alpha}",
                eval_metric=f"Quantile:alpha={alpha}",
                iterations=1800,
                depth=8,
                learning_rate=0.04,
                l2_leaf_reg=5.0,
                random_seed=SEED + 1,
                allow_writing_files=False,
                verbose=False,
                thread_count=-1,
            )
            model.fit(
                X_stress_train,
                y_stress_train,
                cat_features=cat_feature_indices,
                eval_set=(X_stress_test, y_stress_valid),
                early_stopping_rounds=120,
                verbose=False,
            )
            stress_models[alpha] = model
        stress_seconds = time.perf_counter() - start
        stress_q50 = np.maximum(invert_target(stress_models[0.50].predict(X_stress_test)), 0)
        stress_q90_raw = invert_target(stress_models[0.90].predict(X_stress_test))
        stress_model_label = "catboost_product_disjoint"
    elif selected_model_name == "lightgbm_quantile":
        stress_num_medians, stress_cat_levels = fit_preprocessor(stress_train_df)
        X_stress_train = transform_lgbm(stress_train_df, stress_num_medians, stress_cat_levels)
        X_stress_test = transform_lgbm(stress_test_df, stress_num_medians, stress_cat_levels)
        stress_models, stress_seconds = train_lgbm_quantiles(
            X_stress_train, y_stress_train, X_stress_test, y_stress_valid, seed=SEED + 1,
        )
        stress_q50 = np.maximum(invert_target(stress_models[0.50].predict(X_stress_test)), 0)
        stress_q90_raw = invert_target(stress_models[0.90].predict(X_stress_test))
        stress_model_label = "lightgbm_product_disjoint"
    else:
        stress_seconds = 0.0
        stress_q50 = np.full(len(stress_test_df), np.nan)
        stress_q90_raw = np.full(len(stress_test_df), np.nan)
        stress_model_label = "cohort_baseline_not_applicable_to_product_disjoint"

    stress_q90 = np.maximum(stress_q90_raw, stress_q50)
    stress_metrics = metric_row(
        stress_model_label,
        y_stress_test,
        stress_q50,
        stress_q90,
        np.ones(len(y_stress_test), dtype=bool),
        stress_seconds,
        float(np.mean(stress_q90_raw < stress_q50)) if np.isfinite(stress_q90_raw).any() else np.nan,
    )
    stress_result.update(stress_metrics)
    stress_result["calibration_passed"] = bool(
        pd.notna(stress_result.get("q90_coverage"))
        and abs(float(stress_result["q90_coverage"]) - 0.90) <= COVERAGE_TOLERANCE
    )
else:
    stress_result["status"] = "SKIPPED: insufficient rows or RUN_PRODUCT_DISJOINT_STRESS=False"

display(pd.Series(stress_result, name="value").to_frame())


## 11. Score one target safely

Edit TARGET below. Cohort references always come from the historical training window. If any mandatory gate fails, the function returns ABSTAINED and no deviation/exposure calculation.


In [ ]:
def predict_quantiles_for_frame(frame: pd.DataFrame):
    if selected_model_name == "cohort_baseline":
        level, indices, sufficient = find_comparable_indices(frame.iloc[0])
        if not sufficient:
            raise RuntimeError("Comparable cohort is insufficient for baseline prediction.")
        prices = train_df.loc[indices, "unit_price"].to_numpy(float)
        q50 = np.array([np.quantile(prices, 0.50)])
        q90 = np.array([np.quantile(prices, 0.90)])
    elif selected_model_name == "catboost_quantile":
        X = transform_catboost(frame, CAT_NUMERIC_MEDIANS)
        q50 = invert_target(cat_models[0.50].predict(X))
        q90 = invert_target(cat_models[0.90].predict(X))
    else:
        X = transform_lgbm(frame, NUMERIC_MEDIANS, CATEGORY_LEVELS)
        q50 = invert_target(lgb_models[0.50].predict(X))
        q90 = invert_target(lgb_models[0.90].predict(X))
    q50 = np.maximum(q50, 0)
    q90 = np.maximum(q90, q50)
    return q50, q90


def normalize_target(target: dict) -> pd.DataFrame:
    row = {c: target.get(c, np.nan) for c in df.columns}
    row.update(target)
    frame = pd.DataFrame([row])
    frame["observed_at"] = pd.to_datetime(frame["observed_at"], errors="coerce", utc=True)
    for col in ["unit_price", "quantity"] + NUMERIC_CANDIDATES:
        if col in frame:
            frame[col] = pd.to_numeric(frame[col], errors="coerce")
    if pd.isna(frame.loc[0, "observed_at"]):
        raise ValueError("observed_at wajib valid.")
    frame["observed_year"] = frame["observed_at"].dt.year.astype(float)
    frame["observed_month"] = frame["observed_at"].dt.month.astype(float)
    frame["observed_quarter"] = frame["observed_at"].dt.quarter.astype(float)
    frame["time_index_days"] = (
        (frame["observed_at"] - df["observed_at"].min()).dt.total_seconds() / 86400
    )
    return frame


def score_target(target: dict) -> dict:
    frame = normalize_target(target)
    row = frame.iloc[0]
    cohort_level, indices, sufficient = find_comparable_indices(row)
    reasons = abstention_reasons(row, len(indices))
    selected_coverage = benchmark.loc[selected_model_name, "q90_coverage"]
    if pd.isna(selected_coverage) or abs(float(selected_coverage) - 0.90) > COVERAGE_TOLERANCE:
        reasons.append("model interval is not calibrated for the evaluated category/time period")
    training_skus = {
        str(value).strip().lower()
        for value in train_df["manufacturer_part_number"].dropna()
        if known(value)
    }
    target_sku = str(row.get("manufacturer_part_number", "")).strip().lower()
    if (
        target_sku
        and target_sku not in training_skus
        and stress_result.get("calibration_passed") is False
    ):
        reasons.append("model interval is not calibrated for an unseen product")
    quote = pd.to_numeric(pd.Series([row.get("unit_price")]), errors="coerce").iloc[0]
    quantity = pd.to_numeric(pd.Series([row.get("quantity")]), errors="coerce").iloc[0]

    if pd.isna(quote) or quote <= 0:
        reasons.append("price source is zero or invalid")
    if reasons:
        return {
            "status": "ABSTAINED",
            "abstention_reason": "; ".join(dict.fromkeys(reasons)),
            "model": selected_model_name,
            "cohort_level": cohort_level,
            "comparator_count": len(indices),
            "q50": None,
            "q90": None,
            "relative_deviation": None,
            "estimated_exposure_requiring_verification": None,
        }

    q50, q90 = predict_quantiles_for_frame(frame)
    q50, q90 = float(q50[0]), float(q90[0])
    relative_deviation = max(0.0, (float(quote) - q90) / q90)
    exposure = None if pd.isna(quantity) or quantity <= 0 else float(quantity) * max(0.0, float(quote) - q90)
    return {
        "status": "SCORED",
        "model": selected_model_name,
        "cohort_level": cohort_level,
        "comparator_count": len(indices),
        "q50": q50,
        "q90": q90,
        "quoted_unit_price": float(quote),
        "relative_deviation": relative_deviation,
        "estimated_exposure_requiring_verification": exposure,
        "alternative_explanations": [
            "longer warranty", "installation/configuration", "remote-region shipping",
            "urgent delivery", "tax basis mismatch", "bundled software/service",
            "higher-quality component not extracted", "small quantity", "stale comparator",
        ],
    }


# Default example: first out-of-time row. Replace with your own canonical target.
TARGET = test_df.iloc[0].to_dict()
target_result = score_target(TARGET)
display(pd.Series(target_result, name="value").to_frame())


## 12. Export reproducible artefacts

Export menyimpan kedua model LightGBM, optional CatBoost challenger, preprocessing metadata, metrics, out-of-time predictions, model card, source hash, dan resolved package versions. Model belum layak dipakai operasional bila Q90 tidak calibrated, cohort/gate tidak cukup, atau source locators belum tersedia.


In [ ]:
for alpha, model in lgb_models.items():
    joblib.dump(model, ARTIFACT_DIR / f"lightgbm_q{int(alpha * 100)}.joblib")

if RUN_CATBOOST:
    for alpha, model in cat_models.items():
        model.save_model(str(ARTIFACT_DIR / f"catboost_q{int(alpha * 100)}.cbm"))

preprocessing = {
    "features": FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_medians": NUMERIC_MEDIANS,
    "category_levels": CATEGORY_LEVELS,
    "column_mapping": COLUMN_MAPPING,
    "use_log_target": USE_LOG_TARGET,
    "minimum_comparators": MIN_COMPARATORS,
    "cohort_levels": [{"name": name, "fields": columns} for name, columns in COHORT_LEVELS],
}
(ARTIFACT_DIR / "preprocessing.json").write_text(
    json.dumps(preprocessing, indent=2, default=str), encoding="utf-8"
)

validation_benchmark.to_csv(ARTIFACT_DIR / "benchmark_validation.csv")
benchmark.to_csv(ARTIFACT_DIR / "benchmark_out_of_time.csv")
operational_benchmark.to_csv(ARTIFACT_DIR / "benchmark_operational_gate.csv")
evaluation.to_csv(ARTIFACT_DIR / "predictions_out_of_time.csv", index=False)
quality_report.to_csv(ARTIFACT_DIR / "data_quality_report.csv")

resolved_versions = {}
for package in ["lightgbm", "catboost", "pandas", "numpy", "scikit-learn", "joblib"]:
    try:
        resolved_versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        pass
(ARTIFACT_DIR / "resolved_versions.json").write_text(
    json.dumps(resolved_versions, indent=2), encoding="utf-8"
)

lgb_coverage = benchmark.loc["lightgbm_quantile", "q90_coverage"]
selected_coverage = benchmark.loc[selected_model_name, "q90_coverage"]
model_card = {
    "model_name": "SARING Price Engine quantile model",
    "selected_model": selected_model_name,
    "selection_reason": selection_reason,
    "purpose": "Estimate conditional Q50/Q90 price references for comparable laptop observations.",
    "prohibited_claims": [
        "fraud probability", "state loss", "automated procurement decision",
    ],
    "dataset": {
        "file_name": SOURCE_PATH.name,
        "sha256": RAW_SHA256,
        "usable_rows": len(df),
        "train_rows": len(train_df),
        "validation_rows": len(valid_df),
        "out_of_time_test_rows": len(test_df),
    },
    "validation_for_selection": validation_benchmark.reset_index().to_dict(orient="records"),
    "evaluation": benchmark.reset_index().to_dict(orient="records"),
    "operational_gate_evaluation": operational_benchmark.reset_index().to_dict(orient="records"),
    "product_disjoint_stress": stress_result,
    "calibration_gate": {
        "target_q90_coverage": 0.90,
        "tolerance": COVERAGE_TOLERANCE,
        "selected_q90_coverage": selected_coverage,
        "passed": bool(
            pd.notna(selected_coverage)
            and abs(float(selected_coverage) - 0.90) <= COVERAGE_TOLERANCE
        ),
    },
    "abstention_rules": [
        "unresolved SKU/specification", "insufficient comparable cohort",
        "unknown material tax/warranty/shipping basis", "unresolved currency/unit",
        "zero or invalid price", "uncalibrated category/time interval",
    ],
    "source_locator_note": (
        "Training rows retain record_id only. Production engine results require a "
        "resolvable contracts/v1 evidence locator with immutable source hash."
    ),
}
(ARTIFACT_DIR / "model_card.json").write_text(
    json.dumps(model_card, indent=2, default=str), encoding="utf-8"
)

archive_path = shutil.make_archive("/content/saring_price_engine_artifacts", "zip", ARTIFACT_DIR)
print(f"Created: {archive_path}")
print("Files:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(" -", path.name, f"({path.stat().st_size:,} bytes)")

try:
    from google.colab import files
    files.download(archive_path)
except ImportError:
    print("Not in Colab; download manually from:", archive_path)


## Interpretation checklist

Sebelum mempertahankan LightGBM atau challenger:

- LightGBM harus mengalahkan comparable-cohort baseline pada out-of-time pinball loss tanpa merusak Q90 coverage.
- CatBoost dipilih hanya bila peningkatan kualitas melewati threshold material dan biaya training/model masuk akal.
- Periksa coverage per price band, region, category, dan service-bundle context; angka agregat saja tidak cukup.
- Product-disjoint stress harus dilaporkan sebagai stress test, bukan dicampur dengan out-of-time model selection.
- Dataset training harus observed-only dan leakage-safe.
- Scoring operasional harus abstain jika comparable cohort, basis tax/shipping/warranty, unit/currency, atau calibration tidak memadai.
- Production engine tetap harus mengeluarkan exclusions, alternative explanations, dan evidence locators sesuai contracts/v1.
